# EXP-02 live eval — ingest `kb_ddxplus.json`, retrieve, compare to experiment

End-to-end check that the **production RAG stack** (OpenSearch + BGE + hybrid RRF) matches
[EXP-02](../experiments/exp02/README.md) offline benchmarks.

## Pipeline
1. Load `data/kb/kb_ddxplus.json` → `DiseaseDocument`
2. Ingest via `RAGService` (embed + bulk upsert) — skip when `EXP02_REINGEST=false`
3. Build patient queries exactly like `experiments/exp02/eval_retrieval.py`
4. Retrieve with `Retriever` (BM25 + dense kNN + hybrid RRF)
5. Report Hit@1 / Hit@3 / Hit@5 / MRR vs committed baselines
6. Save to `experiments/exp02/results/live_eval_latest.json` (reload without re-running eval)

## Prerequisites
- `.env` with `OPENSEARCH_*` credentials
- Index migrated: `uv run python -m src.migrations.migrate_ddxplus_index upgrade`
- DDXPlus validate patients (not in repo — licensed data):
  - Download `release_validate_patients` from DDXPlus Figshare
  - Place at `data/eval/release_validate_patients` (CSV)
  - Place `release_evidences.json` at `data/eval/release_evidences.json`
- Optional: `export EXP02_DATA=/path/to/eval/inputs`

## Performance / OpenSearch client

The sync client uses **urllib3** (`Urllib3HttpConnection`). For parallel eval:

| Setting | Default | Notes |
|---------|---------|-------|
| `EXP02_WORKERS` | `8` | Concurrent searches via `asyncio.to_thread` |
| `OPENSEARCH_POOL_MAXSIZE` | `16` in `.env` | Must be ≥ `EXP02_WORKERS` or urllib3 logs `Connection pool is full` |
| `OPENSEARCH_TIMEOUT` | `60` | Raise for slow remote bulk/search |
| `EXP02_SAMPLE` | **`132448`** (full validate) | Set `5000` for a quick smoke run |
| `EXP02_PROGRESS_EVERY` | `5000` | Progress log interval (full run ≈ 27 lines/mode) |
| `EXP02_REINGEST` | `false` | Set `true` only when refreshing the 49-doc KB |
| `EXP02_RUN_EVAL` | `true` | Set `false` to skip the long eval cell and load saved results |
| `EXP02_RESULTS` | `experiments/exp02/results/live_eval_latest.json` | Override path for save/load |

**Full validate (132,448 patients):** three modes (BM25 + dense + hybrid) ≈ **3–4 hours** on a typical Aiven setup (observed ~3.7 h with `EXP02_WORKERS=8`). Use `EXP02_REINGEST=false` if the index is already loaded. Overnight run recommended.

Restart the kernel after changing `OPENSEARCH_*` pool/timeout settings (client singleton).

Query construction uses `normalize_symptom_phrase` from `src.services.rag.preprocess` — same
normalizer as `build_kb.py`. Preprocessing is **disabled** on the retriever (`Exp02Preprocess`)
so queries match EXP-02 BM25/hybrid baselines.

## Baseline parity (what matches offline EXP-02)

| Mode | Fair comparison? | Why |
|------|------------------|-----|
| **BM25** | Yes | Same space-joined phrase text; live uses OpenSearch `match` on `keyword_text` |
| **Hybrid RRF** | Yes | Same query text; live uses OpenSearch RRF pipeline vs offline local fusion |
| **Dense kNN** | **Approximate only** | Offline `eval_dense_hybrid.py` uses legacy `embed_text_query()` — see Notes |

**Doc embeddings** (ingest) match offline no-desc: `build_embed_text` via `DiseaseDocument`, no description in the vector.

**Query embeddings** for vector/hybrid use production `TextEmbeddingService.embed_query()` / `embed_queries()` (space-joined phrases + BGE query prefix). Offline dense uses a different template — do not expect Hit@1 to match the 83.11% dense baseline closely.

**Latest committed live run** (full validate, 2026-06-26): BM25 98.65% · dense 79.54% · hybrid 92.30% Hit@1 — see `experiments/exp02/results/live_eval_latest.json`.

In [9]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "services" / "rag").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "pyproject.toml").exists(), "Run from repo root or notebooks/"
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: /home/mitu/projects/disease-diagnosis-rag-system


In [ ]:
import ast
import csv
import json
import random
import zipfile
from collections import defaultdict
from dataclasses import dataclass

from src.db.vector_db.opensearch import get_opensearch_client
from src.services.inference.embeddings.service import TextEmbeddingService
from src.services.rag.pipeline import RAGService
from src.services.rag.preprocess import PreprocessPipeline, normalize_symptom_phrase
from src.services.rag.retrieve import Retriever
from src.services.rag.schemas import (
    Bm25RetrieveRequest,
    DiseaseDocument,
    HybridRetrieveRequest,
    VectorRetrieveRequest,
)
from src.settings import settings

# --- eval config (match experiments/exp02 full validate) ---
FULL_VALIDATE_N = 132_448  # DDXPlus validate split size in EXP-02
SAMPLE = int(os.environ.get("EXP02_SAMPLE", str(FULL_VALIDATE_N)))  # 0 = all CSV rows
SEED = 13
EVAL_TOP_K = 49  # full corpus — needed for correct Hit@5 / MRR
EVAL_WORKERS = int(os.environ.get("EXP02_WORKERS", "8"))  # keep <= OPENSEARCH_POOL_MAXSIZE
PROGRESS_EVERY = int(os.environ.get("EXP02_PROGRESS_EVERY", "5000"))
REINGEST = os.environ.get("EXP02_REINGEST", "false").lower() in ("1", "true", "yes")
RUN_EVAL = os.environ.get("EXP02_RUN_EVAL", "true").lower() in ("1", "true", "yes")
RESULTS_PATH = Path(
    os.environ.get(
        "EXP02_RESULTS",
        PROJECT_ROOT / "experiments/exp02/results/live_eval_latest.json",
    )
)

KB_PATH = PROJECT_ROOT / "data/kb/kb_ddxplus.json"
BASELINE_BM25 = PROJECT_ROOT / "experiments/exp02/results/EXP02_eval_report.md"
BASELINE_HYBRID = PROJECT_ROOT / "experiments/exp02/results/dense_hybrid_summary.md"

EXP02_DATA = Path(os.environ.get("EXP02_DATA", PROJECT_ROOT / "data/eval"))
EVID_PATH = EXP02_DATA / "release_evidences.json"
PATIENTS_PATH = EXP02_DATA / "release_validate_patients"

for zip_path in (
    EXP02_DATA / "release_validate_patients.zip",
    EXP02_DATA / "eval" / "release_validate_patients.zip",
):
    if not PATIENTS_PATH.exists() and zip_path.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(EXP02_DATA)
        break

print("Index alias:", settings.RETRIEVE_INDEX_ALIAS)
print("Search pipeline:", settings.CURRENT_SEARCH_PIPELINE)
print("KB:", KB_PATH)
print("Patients:", PATIENTS_PATH, "exists:", PATIENTS_PATH.exists())
print("Evidences:", EVID_PATH, "exists:", EVID_PATH.exists())
sample_label = "all CSV rows" if SAMPLE == 0 else str(SAMPLE)
print(f"Sample size: {sample_label} (SEED={SEED} when subsampling)")
print(f"Workers={EVAL_WORKERS}, progress every {PROGRESS_EVERY}, REINGEST={REINGEST}")
if SAMPLE == 0 or SAMPLE >= FULL_VALIDATE_N:
    print(
        f"⚠ Full validate eval (~{FULL_VALIDATE_N} patients × 3 modes). "
        "Expect several hours. Set EXP02_REINGEST=false if index already loaded."
    )

Index alias: diseases
Search pipeline: hybrid-rrf
KB: /home/mitu/projects/disease-diagnosis-rag-system/data/kb/kb_ddxplus.json
Patients: /home/mitu/projects/disease-diagnosis-rag-system/data/eval/release_validate_patients exists: True
Evidences: /home/mitu/projects/disease-diagnosis-rag-system/data/eval/release_evidences.json exists: True
Sample size: 132448 (SEED=13 when subsampling)
Workers=8, progress every 5000, REINGEST=False
⚠ Full validate eval (~132448 patients × 3 modes). Expect several hours. Set EXP02_REINGEST=false if index already loaded.


## Step 1 — Load KB and ingest

Loads all 49 documents from `kb_ddxplus.json` and upserts through `RAGService.ingest()`.

In [11]:
def kb_row_to_document(row: dict) -> DiseaseDocument:
    return DiseaseDocument(
        doc_id=row["doc_id"],
        disease=row["disease"],
        symptoms=row["symptoms"],
        antecedents=row["antecedents"],
        severity=row["severity"],
        description=row["description"],
        source=row["source"],
    )


kb_rows = json.loads(KB_PATH.read_text(encoding="utf-8"))
records = [kb_row_to_document(row) for row in kb_rows]
disease_to_doc_id = {row["disease"]: row["doc_id"] for row in kb_rows}

print(f"Loaded {len(records)} KB documents")
print(f"Sample doc_id: {records[0].doc_id} — embed_text[:80]: {records[0].embed_text[:80]}...")

embed_service = TextEmbeddingService()
rag = RAGService(embed_service=embed_service)
if REINGEST:
    count = rag.ingest(records)
    print(f"Ingested {count} docs into alias '{settings.RETRIEVE_INDEX_ALIAS}'")
else:
    print("Skipping ingest (REINGEST=False)")

Loaded 49 KB documents
Sample doc_id: J93 — embed_text[:80]: Disease: Spontaneous pneumothorax. Symptoms: pain present, pain related to consu...
Skipping ingest (REINGEST=False)


In [12]:
client = get_opensearch_client()
body = {"size": 0, "track_total_hits": True, "query": {"match_all": {}}}
resp = client.query(settings.RETRIEVE_INDEX_ALIAS, body)
total = resp.hits.total.value if hasattr(resp.hits.total, "value") else int(resp.hits.total)
print(f"Documents in '{settings.RETRIEVE_INDEX_ALIAS}': {total} (expected 49)")

Documents in 'diseases': 49 (expected 49)


## Step 2 — Build EXP-02 patient queries

Same query construction as `experiments/exp02/eval_retrieval.py`:
evidence codes → `normalize_symptom_phrase(question_en)` → space-joined tokens.

In [13]:
if not PATIENTS_PATH.exists() or not EVID_PATH.exists():
    raise FileNotFoundError(
        "DDXPlus eval data missing. See notebook intro — need release_validate_patients "
        f"and release_evidences.json under {EXP02_DATA}"
    )

evidences = json.loads(EVID_PATH.read_text(encoding="utf-8"))
code_phrase = {
    code: normalize_symptom_phrase(meta.get("question_en", code))
    for code, meta in evidences.items()
}


def base_code(evidence: str) -> str:
    return evidence.split("_@_")[0]


def patient_query_text(evidences_field: str) -> str:
    try:
        items = ast.literal_eval(evidences_field)
    except (SyntaxError, ValueError):
        items = []
    phrases: list[str] = []
    seen: set[str] = set()
    for ev in items:
        phrase = code_phrase.get(base_code(str(ev)))
        if phrase and phrase not in seen:
            seen.add(phrase)
            phrases.append(phrase)
    return " ".join(phrases)


with PATIENTS_PATH.open(encoding="utf-8") as handle:
    patient_rows = list(csv.DictReader(handle))

if SAMPLE and SAMPLE < len(patient_rows):
    random.seed(SEED)
    patient_rows = random.sample(patient_rows, SAMPLE)
# SAMPLE == 0 or SAMPLE >= len(patient_rows) → use all loaded rows

eval_cases = []
skipped = 0
for row in patient_rows:
    gold = row["PATHOLOGY"]
    if gold not in disease_to_doc_id:
        continue
    query = patient_query_text(row["EVIDENCES"])
    if not query.strip():
        skipped += 1
        continue
    eval_cases.append({"gold_disease": gold, "query": query})

print(f"Eval cases: {len(eval_cases)} (skipped empty queries: {skipped})")
if len(eval_cases) >= FULL_VALIDATE_N * 0.99:
    print(f"Full validate load (~{FULL_VALIDATE_N} expected) — baselines are directly comparable.")
print(f"Example query: {eval_cases[0]['query'][:120]}...")

Eval cases: 132448 (skipped empty queries: 0)
Full validate load (~132448 expected) — baselines are directly comparable.
Example query: poor diet diagnosis of anemia family members who have been diagnosed with anemia pain related to consultation pain chara...


## Step 3 — Retrieve and score

Uses `Retriever` with identity preprocessing (EXP-02 queries are already normalized).

Baselines (full validate, n=132448) from `experiments/exp02/results/`:

| Mode | Hit@1 | MRR | Parity with live |
|------|-------|-----|------------------|
| BM25 (`keyword_text`) | 98.47% | 0.9921 | Direct |
| Dense kNN (no-desc embed) | 83.11% | 0.8911 | **Legacy query embed** — see below |
| Hybrid RRF (no-desc embed) | 91.17% | 0.9511 | Direct (BM25-heavy on this eval) |

**Dense baseline caveat:** offline `eval_dense_hybrid.py` embeds queries with `embed_text_query()`:

```text
Symptoms and antecedents: fever, cough, fatigue.
```

(no BGE query prefix). Live vector/hybrid uses production `embed_query()` on space-joined phrases **with** the BGE prefix (`Represent this sentence for searching relevant passages: ...`). Doc side matches; query side does not — treat dense offline numbers as directional only.

In [ ]:
import asyncio


class Exp02Preprocess(PreprocessPipeline):
    """EXP-02 queries are pre-normalized; skip preprocess_query."""

    def preprocess_query(self, query: str) -> str:
        return query


@dataclass
class EvalMetrics:
    hit1: int = 0
    hit3: int = 0
    hit5: int = 0
    rr_sum: float = 0.0
    n: int = 0

    def add_rank(self, rank: int | None) -> None:
        if rank is None:
            return
        self.n += 1
        if rank == 1:
            self.hit1 += 1
        if rank <= 3:
            self.hit3 += 1
        if rank <= 5:
            self.hit5 += 1
        self.rr_sum += 1.0 / rank

    def as_dict(self) -> dict[str, float | int]:
        if self.n == 0:
            return {"n": 0, "hit@1": 0.0, "hit@3": 0.0, "hit@5": 0.0, "mrr": 0.0}
        return {
            "n": self.n,
            "hit@1": self.hit1 / self.n,
            "hit@3": self.hit3 / self.n,
            "hit@5": self.hit5 / self.n,
            "mrr": self.rr_sum / self.n,
        }


def rank_of_gold(hits, gold_disease: str) -> int | None:
    for hit in hits:
        if hit.disease == gold_disease:
            return hit.rank
    return None


def _eval_one(retriever: Retriever, mode: str, case: dict, embedding: list[float] | None) -> tuple[str, int | None]:
    query = case["query"]
    gold = case["gold_disease"]
    if mode == "bm25":
        result = retriever.search_bm25(Bm25RetrieveRequest(query=query, top_k=EVAL_TOP_K))
    elif mode == "vector":
        result = retriever.search_vector(
            VectorRetrieveRequest(query=query, top_k=EVAL_TOP_K, embedding=embedding)
        )
    elif mode == "hybrid":
        result = retriever.search_hybrid(
            HybridRetrieveRequest(query=query, top_k=EVAL_TOP_K, embedding=embedding)
        )
    else:
        raise ValueError(mode)
    return gold, rank_of_gold(result.hits, gold)


async def run_eval(retriever: Retriever, mode: str, cases: list[dict]) -> tuple[EvalMetrics, dict[str, EvalMetrics]]:
    """Run eval with asyncio.to_thread (same pattern as future FastAPI routes)."""
    metrics = EvalMetrics()
    per_disease: dict[str, EvalMetrics] = defaultdict(EvalMetrics)
    semaphore = asyncio.Semaphore(EVAL_WORKERS)

    embeddings: list[list[float] | None]
    if mode in ("vector", "hybrid"):
        print(f"  {mode}: embedding {len(cases)} queries (batched) ...")
        embeddings = await asyncio.to_thread(
            embed_service.embed_queries, [case["query"] for case in cases]
        )
    else:
        embeddings = [None] * len(cases)

    async def eval_one(case: dict, embedding: list[float] | None) -> tuple[str, int | None]:
        async with semaphore:
            return await asyncio.to_thread(_eval_one, retriever, mode, case, embedding)

    tasks = [
        asyncio.create_task(eval_one(case, embedding))
        for case, embedding in zip(cases, embeddings, strict=True)
    ]

    completed = 0
    for task in asyncio.as_completed(tasks):
        gold, rank = await task
        metrics.add_rank(rank)
        per_disease[gold].add_rank(rank)
        completed += 1
        if completed % PROGRESS_EVERY == 0 or completed == len(cases):
            print(f"  {mode}: {completed}/{len(cases)} ...")

    return metrics, per_disease


def pct(value: float) -> str:
    return f"{100.0 * value:.2f}%"


retriever = Retriever(
    client=client,
    embed_service=embed_service,
    preprocess=Exp02Preprocess(),
)
print(f"Retriever ready (EXP-02 query mode, workers={EVAL_WORKERS})")

Retriever ready (EXP-02 query mode, workers=8)


In [ ]:
import time

if not RUN_EVAL:
    print("Skipping eval (EXP02_RUN_EVAL=false). Run the save/load cell below to view saved results.")
else:
    eval_started = time.perf_counter()
    print(f"Evaluating {len(eval_cases)} cases × 3 modes (workers={EVAL_WORKERS}) ...")

    print("Running BM25 eval ...")
    bm25_started = time.perf_counter()
    bm25_metrics, bm25_per_disease = await run_eval(retriever, "bm25", eval_cases)
    print(f"  BM25 done in {(time.perf_counter() - bm25_started) / 60:.1f} min")

    print("Running dense (kNN) eval ...")
    vector_started = time.perf_counter()
    vector_metrics, vector_per_disease = await run_eval(retriever, "vector", eval_cases)
    print(f"  Dense done in {(time.perf_counter() - vector_started) / 60:.1f} min")

    print("Running hybrid eval ...")
    hybrid_started = time.perf_counter()
    hybrid_metrics, hybrid_per_disease = await run_eval(retriever, "hybrid", eval_cases)
    print(f"  Hybrid done in {(time.perf_counter() - hybrid_started) / 60:.1f} min")
    print(f"Total eval wall time: {(time.perf_counter() - eval_started) / 3600:.2f} h")

    results = {
        "BM25 (live OpenSearch)": bm25_metrics.as_dict(),
        "Dense kNN (live OpenSearch)": vector_metrics.as_dict(),
        "Hybrid RRF (live OpenSearch)": hybrid_metrics.as_dict(),
    }
    eval_wall_hours = (time.perf_counter() - eval_started) / 3600

Evaluating 132448 cases × 3 modes (workers=8) ...
Running BM25 eval ...
  bm25: 5000/132448 ...
  bm25: 10000/132448 ...
  bm25: 15000/132448 ...
  bm25: 20000/132448 ...
  bm25: 25000/132448 ...
  bm25: 30000/132448 ...
  bm25: 35000/132448 ...
  bm25: 40000/132448 ...
  bm25: 45000/132448 ...
  bm25: 50000/132448 ...
  bm25: 55000/132448 ...
  bm25: 60000/132448 ...
  bm25: 65000/132448 ...
  bm25: 70000/132448 ...
  bm25: 75000/132448 ...
  bm25: 80000/132448 ...
  bm25: 85000/132448 ...
  bm25: 90000/132448 ...
  bm25: 95000/132448 ...
  bm25: 100000/132448 ...
  bm25: 105000/132448 ...
  bm25: 110000/132448 ...
  bm25: 115000/132448 ...
  bm25: 120000/132448 ...
  bm25: 125000/132448 ...
  bm25: 130000/132448 ...
  bm25: 132448/132448 ...
  BM25 done in 69.4 min
Running dense (kNN) eval ...
  vector: embedding 132448 queries (batched) ...
  vector: 5000/132448 ...
  vector: 10000/132448 ...
  vector: 15000/132448 ...
  vector: 20000/132448 ...
  vector: 25000/132448 ...
  vector: 

## Step 4 — Save and reload results

After a full eval, metrics are written to `RESULTS_PATH` (default `experiments/exp02/results/live_eval_latest.json`) plus a UTC-stamped snapshot in the same folder.

To **view saved results without re-running eval**: set `EXP02_RUN_EVAL=false`, restart the kernel, run setup through eval-helpers, then run this cell only.

In [18]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

LIVE_EVAL_DIR = RESULTS_PATH.parent
LIVE_EVAL_LATEST = RESULTS_PATH

EXP02_BASELINES = {
    "BM25 (EXP-02 offline)": {"n": 132448, "hit@1": 0.9847, "hit@3": 0.9995, "hit@5": 0.9999, "mrr": 0.9921},
    "Dense kNN (EXP-02 offline, legacy query embed)": {
        "n": 132448,
        "hit@1": 0.8311,
        "hit@3": 0.9441,
        "hit@5": 0.9682,
        "mrr": 0.8911,
    },
    "Hybrid RRF (EXP-02 offline)": {"n": 132448, "hit@1": 0.9117, "hit@3": 0.9889, "hit@5": 0.9961, "mrr": 0.9511},
}


def print_eval_table(live: dict, baselines: dict | None = None) -> None:
    baselines = baselines or EXP02_BASELINES
    print("\n=== Live vs EXP-02 baseline ===")
    print(f"{'config':<52} {'Hit@1':>8} {'Hit@3':>8} {'Hit@5':>8} {'MRR':>8} {'n':>8}")
    for label, row in {**live, **baselines}.items():
        print(
            f"{label:<52} {pct(row['hit@1']):>8} {pct(row['hit@3']):>8} "
            f"{pct(row['hit@5']):>8} {row['mrr']:>8.4f} {row['n']:>8}"
        )


def save_live_eval(live: dict, *, meta: dict | None = None) -> Path:
    payload = {
        "saved_at": datetime.now(timezone.utc).isoformat(),
        "meta": meta or {},
        "live": live,
        "baselines": EXP02_BASELINES,
    }
    LIVE_EVAL_DIR.mkdir(parents=True, exist_ok=True)
    text = json.dumps(payload, indent=2)
    LIVE_EVAL_LATEST.write_text(text, encoding="utf-8")
    stamped = LIVE_EVAL_DIR / f"live_eval_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.json"
    stamped.write_text(text, encoding="utf-8")
    print(f"Saved {LIVE_EVAL_LATEST.relative_to(PROJECT_ROOT)} and {stamped.name}")
    return LIVE_EVAL_LATEST


def load_live_eval(path: Path = LIVE_EVAL_LATEST) -> dict:
    if not path.is_file():
        raise FileNotFoundError(f"No saved eval at {path}; run the eval cell first.")
    return json.loads(path.read_text(encoding="utf-8"))


if "results" in globals():
    meta = {
        "n_cases": len(eval_cases),
        "eval_workers": EVAL_WORKERS,
        "exp02_sample": os.environ.get("EXP02_SAMPLE", ""),
    }
    if "eval_wall_hours" in globals():
        meta["wall_hours"] = round(eval_wall_hours, 2)
    save_live_eval(results, meta=meta)

payload = load_live_eval(LIVE_EVAL_LATEST)
if payload.get("meta"):
    print(f"Meta: {payload['meta']}  (saved {payload.get('saved_at')})")
print_eval_table(payload["live"], payload.get("baselines"))

Saved experiments/exp02/results/live_eval_latest.json and live_eval_20260626T084341Z.json
Meta: {'n_cases': 132448, 'eval_workers': 8, 'exp02_sample': ''}  (saved 2026-06-26T08:43:41.173478+00:00)

=== Live vs EXP-02 baseline ===
config                                                  Hit@1    Hit@3    Hit@5      MRR        n
BM25 (live OpenSearch)                                 98.65%   99.96%   99.99%   0.9931   132448
Dense kNN (live OpenSearch)                            79.54%   93.60%   96.75%   0.8709   132448
Hybrid RRF (live OpenSearch)                           92.30%   98.85%   99.54%   0.9565   132448
BM25 (EXP-02 offline)                                  98.47%   99.95%   99.99%   0.9921   132448
Dense kNN (EXP-02 offline, legacy query embed)         83.11%   94.41%   96.82%   0.8911   132448
Hybrid RRF (EXP-02 offline)                            91.17%   98.89%   99.61%   0.9511   132448


In [16]:
AFFECTED = {
    "Anemia", "Bronchiectasis", "Cluster headache", "GERD",
    "Guillain-Barré syndrome", "Localized edema", "Panic attack",
    "Pulmonary embolism", "Tuberculosis",
}

print("=== Lowest BM25 Hit@1 diseases (live) ===")
rows = []
for disease, metrics in bm25_per_disease.items():
    d = metrics.as_dict()
    if d["n"]:
        rows.append((d["hit@1"], disease, d["n"], d["mrr"]))
rows.sort()
for hit1, disease, n, mrr in rows[:10]:
    flag = "⚠️" if disease in AFFECTED else ""
    print(f"{pct(hit1):>7}  MRR={mrr:.4f}  n={n:>5}  {disease} {flag}")

=== Lowest BM25 Hit@1 diseases (live) ===
 59.11%  MRR=0.7956  n= 1866  Acute rhinosinusitis 
 85.12%  MRR=0.9244  n= 2748  Unstable angina 
 93.92%  MRR=0.9677  n= 1547  Myocarditis 
 96.92%  MRR=0.9846  n= 3407  Acute laryngitis 
 97.49%  MRR=0.9873  n= 8656  URTI 
 97.59%  MRR=0.9880  n= 1579  SLE 
 98.50%  MRR=0.9925  n= 2340  Stable angina 
 99.05%  MRR=0.9936  n= 2841  Cluster headache ⚠️
 99.20%  MRR=0.9948  n= 2632  Inguinal hernia 
 99.44%  MRR=0.9967  n= 8246  Viral pharyngitis 


## Notes

- **Sample vs full**: default **`EXP02_SAMPLE=132448`** (full validate, same n as offline baselines). Quick smoke:
  `export EXP02_SAMPLE=5000` before starting the kernel.
- **Re-ingest**: default `EXP02_REINGEST=false`. Set `EXP02_REINGEST=true` only when refreshing the 49-doc KB.
- **Skip eval, reload results**: `export EXP02_RUN_EVAL=false`, run setup → eval-helpers → save/load cell.
  Override path with `EXP02_RESULTS=/path/to/live_eval_....json`.
- **Speed**: vector/hybrid pre-embed all queries in batches (`embed_queries`), then run OpenSearch
  searches concurrently via `asyncio.to_thread` + semaphore (`EXP02_WORKERS=8`). Progress every
  `EXP02_PROGRESS_EVERY=5000` rows on full runs. Observed full validate wall time ≈ **3.7 h** (8 workers).
- **Saved artifacts**: `experiments/exp02/results/live_eval_latest.json` + UTC-stamped snapshots; committed
  offline baselines remain in `EXP02_eval_report.md` and `dense_hybrid_summary.md`.
- **BM25 gap**: live OpenSearch BM25 should be close to offline EXP-02 (~98.5% Hit@1 on full set).
  Latest live run: **98.65%** Hit@1 (+0.18 pp vs offline). Small gaps can come from analyzer differences.
- **Hybrid gap**: live hybrid uses OpenSearch RRF pipeline; offline EXP-02 fuses ranks locally.
  Latest live run: **92.30%** Hit@1 (+1.13 pp vs offline 91.17%) — close match expected.
- **Dense kNN gap (expected)**: offline `eval_dense_hybrid.py` does **not** use production embedding:

  | | Offline `embed_text_query()` | Live `embed_query()` / `embed_queries()` |
  |--|------------------------------|------------------------------------------|
  | Query string | `Symptoms and antecedents: {phrases}.` | `{phrase1} {phrase2} ...` (space-joined) |
  | BGE query prefix | No | Yes (`BGE_QUERY_PREFIX`) |
  | Doc embed | no-desc `build_embed_text` (local numpy) | no-desc `build_embed_text` at ingest (OpenSearch kNN) |

  Latest live dense Hit@1 **79.54%** vs offline **83.11%** (~3.6 pp below) — not a production bug by itself.
  Use BM25 + hybrid to validate the stack; re-baseline dense with `embed_query()` offline if you need strict parity.